**FASE 1: limpieza y normalización**
Dadas las derivas del escáner y las fluctuaciones fisiológicas, es necesario limpiar el dataset y reducir el ruido. Además, se realiza una normalización z-score por vóxel (el dataset está únicamente normalizado por run), pues cada voxel tiene una amplitud diferente por cuestiones vasculares; cuanto más cerca de una arteria, más señal tendrá, independientemente de su contenido informativo. 

Puesto que las RDM se construyen con distancias entre patrones multi-vóxel, y para evitar que los vóxeles de mayor amplitud dominen las distancias, se realiza la normalización.

Por otro lado, en el dataset original, Wehbe usó un smoothing Gaussiano de 6mm, una técnica de procesamiento de imágenes que utiliza un filtro reduce el ruido y mejora la relación señal-ruido (SNR). El estudio, sin embargo muestra en el apéndice G que los resultados sin smoothing tienen mayor precisión de clasificación. Además, muchos estudios recientes (Caucheteux & King, 2022; Schrimpf et al., 2021) no aplican smoothing en modelos de codificación y RSA, puesto que preserva la resolución espacial. Por lo tanto, se prescindirá de ello en el presente trabajo.


In [2]:
import scipy.io as sio
import numpy as np
from nilearn.signal import clean

In [3]:
mat = sio.loadmat(f'C:\\Users\\Ale\\Desktop\\TFM\\subject_1.mat', squeeze_me=True)
words = mat['words']
time_info = mat['time']
time_fmri = time_info[:, 0]
runs = time_info[:, 1].astype(int)

word_starts = np.array([w['start'] for w in words])
word_to_tr = np.searchsorted(time_fmri, word_starts) - 1
word_to_tr = np.clip(word_to_tr, 0, len(time_fmri) - 1)

# Guardar los 3 archivos para Colab
np.save('word_to_tr.npy', word_to_tr)
np.save('valid_trs.npy', valid_trs)
np.save('word_texts.npy', np.array(word_texts, dtype=object))

trs_with_words = set(word_to_tr)
fixation_mask = np.array([t not in trs_with_words for t in range(len(time_fmri))])

print(f"TRs de fijación: {fixation_mask.sum()} de {len(fixation_mask)}")

TRs de fijación: 52 de 1351


Este bloque de código identifica los TR de fijación, que serán excluidos del análisis RSE, pero se incluyen en el detrending (el filtro necesita la serie temporal completa para funcionar). El detrending es la eliminación de las tendencias lentas observadas en el notebook 0.

In [4]:
for subj_id in range(1, 9):
    mat = sio.loadmat(f'C:\\Users\\Ale\\Desktop\\TFM\\subject_{subj_id}.mat', squeeze_me=True)
    data = mat['data'].astype(np.float64)
    runs = mat['time'][:, 1].astype(int)

    cleaned_runs = []
    for run_id in np.unique(runs):
        mask = runs == run_id
        run_data = data[mask]

        run_cleaned = clean(
            signals=run_data,
            detrend=True,                      # Eliminar tendencia lineal
            high_pass=0.005,                   # Frecuencia de corte (Hz)
            standardize='zscore_sample',       # Normalizar a z-scores
            t_r=2.0                            # TR en segundos
        )
        cleaned_runs.append(run_cleaned)

    data_clean = np.vstack(cleaned_runs)
    np.save(f'data_subject{subj_id}_clean.npy', data_clean)
    print(f"Sujeto {subj_id}: {data_clean.shape} limpio y guardado")

Sujeto 1: (1351, 37913) limpio y guardado


KeyboardInterrupt: 

In [ ]:
mat = sio.loadmat(f'C:\\Users\\Ale\\Desktop\\TFM\\subject_1.mat', squeeze_me=True)
runs = mat['time'][:, 1].astype(int)
data_clean = np.load('data_subject1_clean.npy')

for run_id in np.unique(runs):
    mask = runs == run_id
    run_data = data_clean[mask]
    print(f"Run {run_id}: media={run_data.mean():.4f}, "
          f"std={run_data.std():.4f}")

Las celdas anteriores se encargan de limpiar y normalizar la señal, operando por runs. Como ha sido explicado anteriormente, entre runs (bloques) el sujeto descansa, el escáner se recalibra y puede darse un cambio en el nivel base de la señal, representado en los gráficos del notebook 0. 

nilearn.signal.clean elimina la tendencia lineal y utiliza un filtro high pass=0.005, como los autores originales. Este filtro de 0.005 hz permite pasar a frecuencias altas (cambios rápidos asociados con el estímulo) sin ser demasiado estricto. 

La segunda celda tiene como propósito confirmar que el z-scoring funciona, cada run tiene media 0 y desviación estándar 1 por vóxel.

In [ ]:
for subj_id in range(1, 9):
    data_clean = np.load(f'data_subject{subj_id}_clean.npy')
    norms = np.sqrt(np.sum(data_clean**2, axis=1))
    threshold = np.mean(norms) + 3 * np.std(norms)
    outliers = np.where(norms > threshold)[0]
    print(f"Sujeto {subj_id}: {len(outliers)} volúmenes outlier "
          f"de {len(norms)} ({len(outliers)/len(norms)*100:.1f}%)")

    np.save(f'outlier_trs_subject{subj_id}.npy', outliers)

Finalmente, esta celda calcula una media global de la señal para cada instante, indicando la actividad en todos los vóxeles a la vez. Si en un TR es mayor de lo normal, como se veía por encima de la línea roja en el notebook 0, se marca como outlier residual. Se corresponde a un artefacto global (movimiento, ruido del escáner, etc.). En lugar de eliminarse, quedan marcados para ser posteriormente utilizados o descartados en el análisis, y comprobar cómo afectan a los resultados, en caso de que lo hagan. Todos los sujetos presentan alrededor de un 2% de outliers.